# Test audio du modèle `hey_zephyr.onnx`

Chargez le modèle ONNX puis une ou plusieurs pistes audio. Le notebook convertit chaque piste en mono 16 kHz, applique le pipeline openWakeWord et affiche le score maximal de détection par fichier.

Dans Google Colab, les cellules vous proposeront de téléverser le modèle puis les audios. En local, indiquez les chemins dans la cellule Paramètres. FFmpeg est fourni automatiquement par la dépendance Python `imageio-ffmpeg`.

Le score est compris entre 0 et 1, mais ce n'est pas une probabilité calibrée. Le seuil affiché est indicatif et doit être évalué avec des exemples positifs et négatifs.

In [5]:
%pip install -q --upgrade openwakeword onnxruntime soundfile scipy pandas imageio-ffmpeg "numpy<2.3"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\tchir\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
from urllib.request import urlretrieve

THRESHOLD = 0.50  # seuil de lecture indicatif
MODEL_PATH = Path('./hey_zephyr.onnx')  # utilisé en local
AUDIO_DIR = Path('./audios')            # utilisé en local

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Environnement :', 'Google Colab' if IN_COLAB else 'Jupyter local')
print(f'Seuil indicatif : {THRESHOLD:.2f}')

if IN_COLAB:
    uploaded_model = files.upload()
    model_candidates = [name for name in uploaded_model if name.lower().endswith('.onnx')]
    if not model_candidates:
        raise FileNotFoundError('Aucun fichier .onnx sélectionné.')
    MODEL_PATH = Path(model_candidates[0]).resolve()
else:
    MODEL_PATH = MODEL_PATH.expanduser().resolve()

if not MODEL_PATH.is_file():
    raise FileNotFoundError(f'Modèle introuvable : {MODEL_PATH}')

# The openWakeWord wheel may omit these shared feature-extraction models.
OWW_MODELS_DIR = Path.home() / '.cache' / 'openwakeword' / 'models'
OWW_MODELS_DIR.mkdir(parents=True, exist_ok=True)
OWW_MODEL_BASE = 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1'
OWW_SHARED_MODELS = {
    'melspectrogram.onnx': OWW_MODELS_DIR / 'melspectrogram.onnx',
    'embedding_model.onnx': OWW_MODELS_DIR / 'embedding_model.onnx',
}

for filename, model_path in OWW_SHARED_MODELS.items():
    if not model_path.is_file() or model_path.stat().st_size < 100_000:
        temporary_path = model_path.with_suffix(model_path.suffix + '.download')
        try:
            print(f'Téléchargement du modèle partagé {filename}...')
            urlretrieve(f'{OWW_MODEL_BASE}/{filename}', temporary_path)
            if temporary_path.stat().st_size < 100_000:
                raise RuntimeError(f'Téléchargement incomplet : {filename}')
            temporary_path.replace(model_path)
        finally:
            temporary_path.unlink(missing_ok=True)
    print(f'Modèle partagé prêt : {model_path.name} ({model_path.stat().st_size / 1024:.0f} Ko)')

import numpy as np
from openwakeword.model import Model

oww = Model(
    wakeword_models=[str(MODEL_PATH)],
    inference_framework='onnx',
    melspec_model_path=str(OWW_SHARED_MODELS['melspectrogram.onnx']),
    embedding_model_path=str(OWW_SHARED_MODELS['embedding_model.onnx']),
)
KEY = next(iter(oww.models))

def clip_score(wav16_int16):
    """Score maximal du modèle sur le clip audio, entre 0 et 1."""
    oww.reset()
    predictions = oww.predict_clip(np.asarray(wav16_int16, dtype=np.int16))
    scores = [float(prediction[KEY]) for prediction in predictions if KEY in prediction]
    return max(scores) if scores else 0.0

print(f'Modèle chargé : {MODEL_PATH.name} ({MODEL_PATH.stat().st_size / 1024:.1f} Ko)')
print('Clé de sortie :', KEY)

Environnement : Jupyter local
Seuil indicatif : 0.50
Téléchargement du modèle partagé melspectrogram.onnx...
Modèle partagé prêt : melspectrogram.onnx (1062 Ko)
Téléchargement du modèle partagé embedding_model.onnx...
Modèle partagé prêt : embedding_model.onnx (1295 Ko)
Modèle chargé : hey_zephyr.onnx (838.9 Ko)
Clé de sortie : hey_zephyr


## Charger les pistes audio

Formats courants : WAV, MP3, M4A, OGG, FLAC, AAC, WMA et WebM. Dans Colab, sélectionnez plusieurs fichiers à la fois. En local, placez-les dans `AUDIO_DIR`.

In [3]:
SUPPORTED_AUDIO = {'.wav', '.mp3', '.m4a', '.ogg', '.flac', '.aac', '.wma', '.webm'}

if IN_COLAB:
    uploaded_audio = files.upload()
    audio_paths = [
        Path(name).resolve()
        for name in uploaded_audio
        if Path(name).suffix.lower() in SUPPORTED_AUDIO
    ]
else:
    audio_root = AUDIO_DIR.expanduser().resolve()
    if not audio_root.is_dir():
        raise FileNotFoundError(f'Dossier audio introuvable : {audio_root}')
    audio_paths = sorted(
        path.resolve() for path in audio_root.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_AUDIO
    )

if not audio_paths:
    raise FileNotFoundError('Aucun fichier audio sélectionné ou trouvé.')

print(f'{len(audio_paths)} piste(s) à tester :')
for audio_path in audio_paths:
    print(' -', audio_path.name)

6 piste(s) à tester :
 - Zephyr_1.m4a
 - Zephyr_2.m4a
 - Zephyr_3.m4a
 - Zephyr_4.m4a
 - Zephyr_5.m4a
 - Zephyr_6.m4a


In [6]:
import os
import subprocess
import tempfile
import imageio_ffmpeg
import soundfile as sf
import pandas as pd

FFMPEG_EXE = imageio_ffmpeg.get_ffmpeg_exe()
print('FFmpeg utilisé :', FFMPEG_EXE)
results = []

for audio_path in audio_paths:
    converted_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as temporary_file:
            converted_path = temporary_file.name

        subprocess.run(
            [FFMPEG_EXE, '-y', '-loglevel', 'error', '-i', str(audio_path),
             '-ac', '1', '-ar', '16000', converted_path],
            check=True, capture_output=True, text=True,
        )

        audio, sample_rate = sf.read(converted_path, dtype='float32')
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sample_rate != 16000:
            raise ValueError(f'Fréquence inattendue après conversion : {sample_rate} Hz')
        if audio.size == 0:
            raise ValueError('Fichier audio vide.')

        audio_int16 = (np.clip(audio, -1.0, 1.0) * 32767).astype(np.int16)
        score = clip_score(audio_int16)
        results.append({
            'fichier': audio_path.name,
            'durée_s': round(len(audio_int16) / 16000, 2),
            'score_max': round(score, 4),
            f'>= {THRESHOLD:.2f} (indicatif)': score >= THRESHOLD,
            'statut': 'OK',
            'erreur': '',
        })
        print(f'{audio_path.name:45s} score max = {score:.3f}')
    except Exception as exc:
        results.append({
            'fichier': audio_path.name,
            'durée_s': None,
            'score_max': None,
            f'>= {THRESHOLD:.2f} (indicatif)': None,
            'statut': 'ERREUR',
            'erreur': str(exc),
        })
        print(f'ERREUR - {audio_path.name}: {exc}')
    finally:
        if converted_path and os.path.exists(converted_path):
            os.remove(converted_path)

results_df = pd.DataFrame(results)
display(results_df)

FFmpeg utilisé : C:\Users\tchir\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe
Zephyr_1.m4a                                  score max = 0.049
Zephyr_2.m4a                                  score max = 0.944
Zephyr_3.m4a                                  score max = 0.985
Zephyr_4.m4a                                  score max = 0.999
Zephyr_5.m4a                                  score max = 0.999
Zephyr_6.m4a                                  score max = 0.999


,fichier,durée_s,score_max,>= 0.50 (indicatif),statut,erreur
0,Zephyr_1.m4a,2.47,0.0492,False,OK,
1,Zephyr_2.m4a,1.96,0.9440,True,OK,
2,Zephyr_3.m4a,2.60,0.9846,True,OK,
3,Zephyr_4.m4a,2.56,0.9991,True,OK,
4,Zephyr_5.m4a,2.56,0.9992,True,OK,
5,Zephyr_6.m4a,2.77,0.9991,True,OK,


In [7]:
csv_path = Path('hey_zephyr_scores.csv')
results_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print('Résultats enregistrés :', csv_path.resolve())
if IN_COLAB:
    files.download(str(csv_path))

Résultats enregistrés : C:\Users\tchir\Downloads\hey_zephyr_scores.csv
